# 08 — Joint Training: Vehicle Detection + Lane Segmentation (v4)

**Goal:** Train a joint YOLO26-S model with:
- Native 5-class vehicle detection head (car, truck, bus, motorcycle, bicycle)
- Transformer-based lane segmentation head (SRA attention)
- Shared backbone/neck, no DualTaskNeck

**Architecture (v4):**
- Backbone: YOLO26-S (pretrained on COCO, detection head replaced with nc=5)
- Lane head: LightMUSTER transformer (embed_dim=128, depth=2, SRA)
- Loss: fixed weighting (det=1.0, lane=0.3)
- Masks: 160x160 square (matching 640x640 input)

## 1 — Setup

In [ ]:
!pip install -q ultralytics>=8.4.0 opencv-python matplotlib pyyaml tqdm
!pip install -q "torchmetrics[detection]"

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu} ({mem:.1f} GB)")
else:
    print("WARNING: No GPU detected — training will be very slow.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"
DATASET_DIR = "/content/bdd100k_yolo"
print(f"EcoCAR root: {ECOCAR_ROOT}")
print(f"Dataset dir: {DATASET_DIR}")

In [ ]:
import sys

# Use the project directory on Drive for imports
PROJECT_DIR = os.path.join(ECOCAR_ROOT, "project")
assert os.path.isdir(PROJECT_DIR), f"Project not found at {PROJECT_DIR}"
sys.path.insert(0, PROJECT_DIR)

from src.utils.class_map import VEHICLE_CLASSES, NUM_VEHICLE_CLASSES
print(f"Vehicle classes ({NUM_VEHICLE_CLASSES}): {VEHICLE_CLASSES}")

## 2 — Configuration

Key design choices:
- **Native nc=5** detection head (no COCO-80 remapping)
- **Square masks** (160x160) match the 640x640 input
- **Transformer lane head** (LightMUSTER with SRA)
- **Fixed loss weights** (det=1.0, lane=0.3) — no staged weighting needed
- **SGD** with low LR for stable fine-tuning

In [ ]:
import yaml

# Detection-only checkpoint from Notebook 03 (warm start)
DET_CKPT = f"{ECOCAR_ROOT}/weights/bdd100k_yolo26s_vehicle_best.pt"

cfg = {
    "run_name": "vehicle_lane_v4",
    "device": "cuda",
    "amp": True,

    "model": {
        "backbone_weights": "yolo26s.pt",
        "lane_head": {
            "type": "transformer",
            "embed_dim": 128,
            "num_heads": 4,
            "depth": 2,
            "hidden_channels": 64,
        },
    },

    "data": {
        "dataset_root": DATASET_DIR,
        "img_size": 640,
        "mask_height": 160,
        "mask_width": 160,
        "batch_size": 16,
        "num_workers": 4,
    },

    "training": {
        "epochs": 40,
        "patience": 8,
    },

    "optimizer": {
        "type": "sgd",
        "lr": 0.002,
        "momentum": 0.937,
        "weight_decay": 5e-4,
        "backbone_lr_scale": 0.1,
    },

    "scheduler": {
        "type": "cosine",
        "warmup_epochs": 3,
        "min_lr_ratio": 0.01,
    },

    "loss": {
        "lane": {
            "type": "bce_dice",
            "bce_weight": 0.5,
            "dice_weight": 0.5,
        },
        "multitask": {
            "strategy": "fixed",
            "det_weight": 1.0,
            "lane_weight": 0.3,
        },
        "detection_preservation": {
            "enabled": False,
        },
    },

    "eval": {
        "conf_thresh": 0.001,
        "nms_iou_thresh": 0.6,
        "max_det": 300,
        "lane_thresholds": [0.3, 0.4, 0.5, 0.6, 0.7],
    },
}

SAVE_DIR = f"{ECOCAR_ROOT}/training_runs/{cfg['run_name']}"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save config
cfg_path = os.path.join(SAVE_DIR, "config.yaml")
with open(cfg_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"Config saved: {cfg_path}")
print(f"Save dir:    {SAVE_DIR}")

## 3 — Build Model

In [ ]:
from src.multitask_model import build_multitask_model

model = build_multitask_model(cfg)
model.print_summary()

In [ ]:
# Warm start from detection-only checkpoint
if os.path.isfile(DET_CKPT):
    loaded = model.warm_start_from_checkpoint(DET_CKPT)
    print(f"Warm start: loaded {loaded} keys from {DET_CKPT}")
else:
    print(f"No detection checkpoint found at {DET_CKPT}")
    print("Training from COCO pretrained backbone only.")

In [ ]:
# Forward-pass sanity check
dummy = torch.randn(1, 3, 640, 640)
if torch.cuda.is_available():
    dummy = dummy.cuda()
    model = model.cuda()

with torch.no_grad():
    out = model(dummy)

print("Output keys:", list(out.keys()))
det = out['det_output']
if isinstance(det, torch.Tensor):
    print(f"  det_output shape: {det.shape}")
elif isinstance(det, (list, tuple)):
    print(f"  det_output: {type(det).__name__} with {len(det)} elements")
    for j, d in enumerate(det):
        if hasattr(d, 'shape'):
            print(f"    [{j}] {d.shape}")
        else:
            print(f"    [{j}] {type(d).__name__}: {d}")
else:
    print(f"  det_output: {type(det)}")
print(f"  lane_logits shape: {out['lane_logits'].shape}")
print(f"  neck_features: {len(out['neck_features'])} scales")
model = model.cpu()
torch.cuda.empty_cache()

## 4 — Dataset

In [ ]:
import tarfile

os.makedirs(DATASET_DIR, exist_ok=True)

# 1. Images + Labels from NB02 tar
NB02_TAR = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_yolo_nb02.tar")
if not os.path.isdir(os.path.join(DATASET_DIR, "images", "val")):
    assert os.path.isfile(NB02_TAR), f"NB02 tar not found: {NB02_TAR}\nRun Notebook 02 first."
    print(f"Extracting {NB02_TAR} ...")
    with tarfile.open(NB02_TAR, "r") as tar:
        tar.extractall(DATASET_DIR, filter='data')
    print("Done.")
else:
    print("Images/labels already extracted.")

# 2. Lane masks from NB07 tar
MASKS_TAR = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_lane_masks.tar")
if not os.path.isdir(os.path.join(DATASET_DIR, "masks", "val")):
    assert os.path.isfile(MASKS_TAR), f"Masks tar not found: {MASKS_TAR}\nRun Notebook 07 first."
    print(f"Extracting {MASKS_TAR} ...")
    with tarfile.open(MASKS_TAR, "r") as tar:
        tar.extractall(DATASET_DIR, filter='data')
    print("Done.")
else:
    print("Masks already extracted.")

# Quick check
for split in ["train", "val"]:
    n_img = len(os.listdir(os.path.join(DATASET_DIR, "images", split)))
    n_lbl = len(os.listdir(os.path.join(DATASET_DIR, "labels", split)))
    mask_dir = os.path.join(DATASET_DIR, "masks", split)
    n_mask = len(os.listdir(mask_dir)) if os.path.isdir(mask_dir) else 0
    print(f"  {split}: {n_img} images, {n_lbl} labels, {n_mask} masks")

In [ ]:
from src.data.transforms import JointTransform
from src.data.dataset import JointBDDDataset, joint_collate_fn
from torch.utils.data import DataLoader

data_cfg = cfg["data"]
root = data_cfg["dataset_root"]

train_transform = JointTransform(
    img_size=data_cfg["img_size"],
    mask_height=data_cfg["mask_height"],
    mask_width=data_cfg["mask_width"],
    augment=True,
)
val_transform = JointTransform(
    img_size=data_cfg["img_size"],
    mask_height=data_cfg["mask_height"],
    mask_width=data_cfg["mask_width"],
    augment=False,
)

train_ds = JointBDDDataset(
    images_dir=os.path.join(root, "images", "train"),
    labels_dir=os.path.join(root, "labels", "train"),
    masks_dir=os.path.join(root, "masks", "train"),
    transform=train_transform,
)
val_ds = JointBDDDataset(
    images_dir=os.path.join(root, "images", "val"),
    labels_dir=os.path.join(root, "labels", "val"),
    masks_dir=os.path.join(root, "masks", "val"),
    transform=val_transform,
)

train_loader = DataLoader(
    train_ds, batch_size=data_cfg["batch_size"], shuffle=True,
    num_workers=data_cfg["num_workers"], collate_fn=joint_collate_fn,
    pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=data_cfg["batch_size"], shuffle=False,
    num_workers=data_cfg["num_workers"], collate_fn=joint_collate_fn,
    pin_memory=True,
)

print(f"Train: {len(train_ds)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_ds)} samples, {len(val_loader)} batches")

## 5 — Sanity Check Batch

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

batch = next(iter(train_loader))

print(f"Images:      {batch['images'].shape}")
print(f"Det targets: {batch['det_targets'].shape}")
print(f"Lane masks:  {batch['lane_masks'].shape}")

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(min(4, batch['images'].shape[0])):
    img = batch['images'][i].permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    mask = batch['lane_masks'][i, 0].numpy()

    axes[0, i].imshow(img)
    axes[0, i].set_title(f"Image {i}")
    axes[0, i].axis('off')

    axes[1, i].imshow(mask, cmap='magma')
    axes[1, i].set_title(f"Lane mask {i}")
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## 6 — Train

The `JointTrainer` handles:
- Fixed multi-task loss (det_weight=1.0, lane_weight=0.3)
- EMA-based evaluation
- Metrics logging and best checkpoint selection

In [ ]:
from src.trainers.trainer import JointTrainer

trainer = JointTrainer(cfg, model, train_loader, val_loader)
trainer.save_dir = SAVE_DIR

print(f"Training for {cfg['training']['epochs']} epochs")
print(f"Save dir: {SAVE_DIR}")

## 7 — Training Results

In [ ]:
import traceback

try:
    history = trainer.train()
except Exception as e:
    print(f"\nTraining stopped: {e}")
    traceback.print_exc(limit=3)
    history = trainer.history

print(f"\nTraining complete. {len(history)} epochs logged.")

In [ ]:
import matplotlib.pyplot as plt

if history and len(history) > 0:
    epochs_range = range(1, len(history) + 1)

    # Extract metrics from list of epoch dicts
    train_losses = [h.get('train_loss', 0) for h in history]
    det_map50 = [h.get('det_map50', 0) for h in history]
    det_map50_95 = [h.get('det_map50_95', 0) for h in history]
    lane_miou = [h.get('lane_miou', 0) for h in history]
    lane_f1 = [h.get('lane_f1', 0) for h in history]
    val_losses = [h.get('val_loss', 0) for h in history]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Loss
    axes[0].plot(epochs_range, train_losses, label='Train')
    axes[0].plot(epochs_range, val_losses, label='Val')
    axes[0].set_title('Total Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    # Detection mAP
    axes[1].plot(epochs_range, [v*100 for v in det_map50], label='mAP@50')
    axes[1].plot(epochs_range, [v*100 for v in det_map50_95], label='mAP@50-95')
    axes[1].set_title('Detection mAP (%)')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    # Lane metrics
    axes[2].plot(epochs_range, [v*100 for v in lane_miou], label='mIoU')
    axes[2].plot(epochs_range, [v*100 for v in lane_f1], label='F1')
    axes[2].set_title('Lane Metrics (%)')
    axes[2].set_xlabel('Epoch')
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=150)
    plt.show()
else:
    print("No training history available.")

## 8 — Save Checkpoint

In [ ]:
import shutil

# Copy best checkpoint to weights dir
weights_dir = os.path.join(SAVE_DIR, "weights")
best_src = os.path.join(weights_dir, "best_joint.pt")
backup_dir = os.path.join(ECOCAR_ROOT, "weights")
os.makedirs(backup_dir, exist_ok=True)

if os.path.isfile(best_src):
    dst = os.path.join(backup_dir, "vehicle_lane_v4_best.pt")
    shutil.copy2(best_src, dst)
    print(f"Best checkpoint copied to: {dst}")
else:
    print(f"best_joint.pt not found in {weights_dir}")
    # Try last checkpoint
    last_src = os.path.join(weights_dir, "last.pt")
    if os.path.isfile(last_src):
        dst = os.path.join(backup_dir, "vehicle_lane_v4_last.pt")
        shutil.copy2(last_src, dst)
        print(f"Last checkpoint copied to: {dst}")

# Save summary
import json
summary = {
    "run_name": cfg["run_name"],
    "epochs": cfg["training"]["epochs"],
    "architecture": "v4_native_vehicle_transformer_lane",
    "nc": 5,
    "classes": VEHICLE_CLASSES,
    "mask_size": [cfg["data"]["mask_height"], cfg["data"]["mask_width"]],
}
if history and len(history) > 0:
    best_det = max(h.get('det_map50_95', 0) for h in history)
    best_lane = max(h.get('lane_miou', 0) for h in history)
    summary["best_scores"] = {
        "det": best_det,
        "lane": best_lane,
        "joint": best_det * 0.6 + best_lane * 0.4,
    }

summary_path = os.path.join(SAVE_DIR, "summary.json")
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Summary saved: {summary_path}")
print("\n" + "=" * 55)
print(" JOINT TRAINING COMPLETE (v4)")
print(f" Classes: {', '.join(VEHICLE_CLASSES)}")
if 'best_scores' in summary:
    print(f" Best det mAP@50-95: {summary['best_scores']['det']*100:.1f}%")
    print(f" Best lane mIoU:     {summary['best_scores']['lane']*100:.1f}%")
print(f" Checkpoint: {backup_dir}")
print("=" * 55)